In [1]:
# NOTE: this requires running the EM stability test in rSLDS_actual
# EM stability test: 1 rSLDS config is run 10x with the same data, 
# to check EM for init and local optima

In [2]:
import os
import numpy as np
import pandas as pd
from math import sqrt
from scipy.stats import t as _t_dist

In [3]:
# --------------------------------------------------------------
# Load files
# --------------------------------------------------------------

def load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files):

    print("\nLoading files")
    
    dfs = {}
    missing = []
    for i in range(1, n_files + 1):
        path = os.path.join(BASE_DIR, FILENAME_PATTERN.format(i))
        if os.path.exists(path):
            print(f"Loading {path}")
            df = pd.read_csv(path)
            df["__file_id__"] = i
            dfs[i] = df
        else:
            missing.append(path)
    
    if missing:
        print("Warning: missing files:")
        for p in missing:
            print("  -", p)
    
    assert len(dfs) > 0, "No input files found. Set BASE_DIR correctly."

    print("\n")
    
    return dfs

In [7]:
# --------------------------------------------------------------
# t-quantile helper
# --------------------------------------------------------------

def _t_quantile(p, df_):
    return _t_dist.ppf(p, df_)

def em_stability_summary(
    dfs,
    cfg,
    n_regimes,
    dim_latent,
    value_col,
    alpha=0.05,
):
    """
    Across gridsearch_results1..10:
      - select rows matching (cfg, n_regimes, dim_latent)
      - within each file: clean value_col, take mean over matching rows
      - treat those per-file means as EM runs
    Returns:
      DataFrame with columns:
        config, n_regimes, dim_latent, mean, ci_low, ci_high
    """
    run_vals = []

    for i, df in dfs.items():
        dfi = df[
            (df["config"] == cfg) &
            (df["n_regimes"] == n_regimes) &
            (df["dim_latent"] == dim_latent)
        ]
        if dfi.empty:
            continue

        if value_col not in dfi.columns:
            raise KeyError(f"Column '{value_col}' missing in file {i}.")

        vals = (dfi[value_col]
                .replace([np.inf, -np.inf], np.nan)
                .dropna())
        if len(vals) == 0:
            continue

        run_vals.append(vals.mean())

    run_vals = np.array(run_vals, dtype=float)
    run_vals = run_vals[~np.isnan(run_vals)]
    n = len(run_vals)

    if n == 0:
        raise ValueError("No valid observations for requested config triple across files.")

    mean = run_vals.mean()
    if n > 1:
        std = run_vals.std(ddof=1)
        crit = _t_quantile(1 - alpha / 2, n - 1)
        half_width = crit * std / sqrt(n)
        ci_low = mean - half_width
        ci_high = mean + half_width
    else:
        ci_low = ci_high = mean

    return pd.DataFrame([{
        "config": cfg,
        "n_regimes": n_regimes,
        "dim_latent": dim_latent,
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }])


In [10]:
# --------------------------------------------------------------
# Create table: jackknife unrestricted
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/jackknife unrestricted"
FILENAME_PATTERN = "gridsearch_results_jk{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=15)

targets = [
    ("[y]", 6, 1),
    ("[g,v]", 6, 2),]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk1.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk2.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk3.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk4.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk5.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk6.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk7.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk8.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk9.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted/gridsearch_results_jk10.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife unrestricted

,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,[y],6,1,0.067,0.064,0.070
1,"[g,v]",6,2,0.042,0.040,0.043


In [18]:
# --------------------------------------------------------------
# Create table: jackknife restricted
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/jackknife restricted"
FILENAME_PATTERN = "gridsearch_results_jk{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=15)

targets = [
    ("factor2_ff3", 4, 3),
    ("factor2_ff3mom", 4, 4),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk1.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk2.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk3.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk4.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk5.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk6.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk7.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk8.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk9.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk10.csv
Loading /Users/chrismader/Python/SLDS/Out/jackknife restricted/gridsearch_results_jk

,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,factor2_ff3,4,3,0.015,0.013,0.017
1,factor2_ff3mom,4,4,0.015,0.013,0.017


In [16]:
# --------------------------------------------------------------
# Create table: stability y61
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/stability y61"
FILENAME_PATTERN = "gridsearch_results{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=10)

targets = [
    ("[y]", 6, 1),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results1.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results2.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results3.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results4.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results5.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results6.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results7.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results8.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results9.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results10.csv




,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,[y],6,1,0.005,0.003,0.007


In [15]:
# --------------------------------------------------------------
# Create table: stability gv62
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/stability gv62"
FILENAME_PATTERN = "gridsearch_results{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=10)

targets = [
    ("[g,v]", 6, 2),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results1.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results2.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results3.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results4.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results5.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results6.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results7.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results8.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results9.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results10.csv




,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,"[g,v]",6,2,0.043,0.041,0.045


In [29]:
import pandas as pd

CSV_PATH = "DRO/gridsearch_results.csv"

def summarize_config_dim(
    config,
    dim_latent: int,
    csv_path: str = CSV_PATH,
    total_securities: int = 75,
):
    """
    For a given config and dim_latent:

    1) Filter the grid-search results.
    2) For EACH security, pick the row (across all n_regimes) with the
       highest cagr_rel_ex_ante (per-security best n_regimes).
    3) Bucket each security into Excellent / Good / Fair / Poor.
    4) Return a one-row summary table with counts and percentages.

    config can be:
      - "[g,v]", "[y]", "fund2", etc. (string, with or without spaces)
      - ["g", "v"] (list or tuple)
    """

    # normalize config into a compact string without spaces, e.g. "[g,v]"
    if isinstance(config, (list, tuple)):
        config_str = "[" + ",".join(str(c).strip() for c in config) + "]"
    else:
        config_str = str(config).strip()
    config_key = config_str.replace(" ", "")

    df = pd.read_csv(csv_path)

    # also normalize the config column (remove spaces)
    config_no_space = df["config"].astype(str).str.replace(" ", "", regex=False)

    # 1) filter to given config + dim_latent
    sub = df[(config_no_space == config_key) & (df["dim_latent"] == dim_latent)]
    if sub.empty:
        raise ValueError(f"No rows found for config={config_str}, dim_latent={dim_latent}")

    # 2) for EACH security, keep the SINGLE row with the max cagr_rel_ex_ante
    #    (this implicitly chooses the "best" n_regimes per security)
    best_per_security = (
        sub.sort_values("cagr_rel_ex_ante", ascending=False)
           .groupby("security", as_index=False)
           .head(1)
    )

    # 3) bucket into Excellent / Good / Fair / Poor
    def bucket(x: float) -> str:
        if x >= 0.05:
            return "Excellent"
        elif x >= 0.02:
            return "Good"
        elif x >= 0.0:
            return "Fair"
        else:
            return "Poor"

    best_per_security["group"] = best_per_security["cagr_rel_ex_ante"].apply(bucket)

    counts = (
        best_per_security["group"]
        .value_counts()
        .reindex(["Excellent", "Good", "Fair", "Poor"], fill_value=0)
    )

    # 4) build one-row summary table with required columns
    summary = pd.DataFrame(
        {
            "Config": [config_str],
            "n_regimes": ["best"],          # per-security best n_regimes
            "dim_latent": [dim_latent],
            "Excellent": [counts["Excellent"]],
            "Good": [counts["Good"]],
            "Fair": [counts["Fair"]],
            "Poor": [counts["Poor"]],
        }
    )

    for col_count, col_pct in [
        ("Excellent", "Excellent_pct"),
        ("Good", "Good_pct"),
        ("Fair", "Fair_pct"),
        ("Poor", "Poor_pct"),
    ]:
        pct = summary[col_count] / float(total_securities) * 100.0
        summary[col_pct] = pct.map(lambda x: f"{x:.1f}%")

    return summary, best_per_security


summary_y, _ = summarize_config_dim(config="[y]", dim_latent=1)
summary_gv, _ = summarize_config_dim(config="[g,v]", dim_latent=2)
summary_ff3, _ = summarize_config_dim(config="factor2_ff3", dim_latent=3)
summary_ff3mom, _ = summarize_config_dim(config="factor2_ff3mom", dim_latent=4)

summary_all = pd.concat(
    [summary_y, summary_gv, summary_ff3, summary_ff3mom],
    ignore_index=True)

print ("SUMMARY")
summary_all

SUMMARY


,Config,n_regimes,dim_latent,Excellent,Good,Fair,Poor,Excellent_pct,Good_pct,Fair_pct,Poor_pct
0,[y],best,1,57,17,0,1,76.0%,22.7%,0.0%,1.3%
1,"[g,v]",best,2,44,24,6,1,58.7%,32.0%,8.0%,1.3%
2,factor2_ff3,best,3,35,25,14,1,46.7%,33.3%,18.7%,1.3%
3,factor2_ff3mom,best,4,28,29,13,5,37.3%,38.7%,17.3%,6.7%
